# ResNet Custom Model for Drone Acoustic Detection

In [2]:
!pip install datasets numpy scikit-learn seaborn librosa tqdm optuna librosa torchcodec

In [1]:
import io
import logging
import os
import random
from dataclasses import dataclass

import librosa
import matplotlib.pyplot as plt
import numpy as np
import optuna
import seaborn as sns
import soundfile as sf
import torch
from datasets import Audio, DatasetDict, load_dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score, 
)
from torch import nn, optim
from torch.utils.data import DataLoader
from tqdm import tqdm

/opt/conda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configurations

In [3]:
SEED = 42
BATCH_SIZE = 32
NUM_WORKERS = 24
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAMPLING_RATE = 16000
EPOCHS = 10
OPTUNA_EPOCHS = 3
N_TRIALS = 25
MODEL_SAVE_PATH = "models/resnet_custom_hyperparameters.pt"

# External optimization target: F1 on Hibou-Foundation/all_tests_ds_3
ALL_TESTS_DATASET_NAME = "Hibou-Foundation/all_tests_ds_3"
ALL_TESTS_BATCH_SIZE = 1
ALL_TESTS_NUM_WORKERS = 1
ALL_TESTS_THRESHOLD_GRID = np.arange(0.05, 0.951, 0.01)
ALL_TESTS_MAX_SAMPLES_PER_SPLIT = None  # Set an int for faster debug runs

# Persistent tracking for long Optuna runs
LOG_DIR = "logs"
HPO_LOG_PATH = os.path.join(LOG_DIR, "resnet_custom_hpo.log")
OPTUNA_STORAGE_PATH = os.path.join(LOG_DIR, "resnet_custom_hpo.sqlite3")
OPTUNA_STORAGE_URL = f"sqlite:///{OPTUNA_STORAGE_PATH}"
OPTUNA_STUDY_NAME = "resnet_custom_all_tests_f1"

# Used only if Optuna is skipped.
FALLBACK_LR = 3e-4
FALLBACK_WEIGHT_DECAY = 1e-1
FALLBACK_MODEL_PARAMS = {
    "channels": (32, 64, 128, 256),
    "num_blocks": (1, 1, 1),
    "dropout": 0.5,
    "fc_hidden": 128,
    "initial_kernel_size": 5,
    "block_kernel_size": 3,
}


def get_hpo_logger(log_path):
    logger = logging.getLogger("resnet_custom_hpo")
    logger.setLevel(logging.INFO)

    if logger.handlers:
        return logger

    formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)

    file_handler = logging.FileHandler(log_path, encoding="utf-8")
    file_handler.setFormatter(formatter)

    logger.addHandler(stream_handler)
    logger.addHandler(file_handler)
    return logger


# Set seeds for reproducibility between runs
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

os.makedirs(LOG_DIR, exist_ok=True)
LOGGER = get_hpo_logger(HPO_LOG_PATH)
LOGGER.info("Logger initialized. File path: %s", HPO_LOG_PATH)
LOGGER.info("Optuna storage path: %s", OPTUNA_STORAGE_PATH)

print(f"Using device: {DEVICE}")
print(f"Number of workers: {NUM_WORKERS}")
print(f"Logging to: {HPO_LOG_PATH}")
print(f"Optuna storage: {OPTUNA_STORAGE_PATH}")

2026-02-13 16:01:29 | INFO | Logger initialized. File path: logs/resnet_custom_hpo.log
2026-02-13 16:01:29 | INFO | Optuna storage path: logs/resnet_custom_hpo.sqlite3


Using device: cuda
Number of workers: 24
Logging to: logs/resnet_custom_hpo.log
Optuna storage: logs/resnet_custom_hpo.sqlite3


## Dataset & Collate function

In [4]:
dataset = load_dataset("Hibou-Foundation/ds_121_melspecto_noaug_balanced_chunked")
dataset = dataset.with_format("torch", columns=["audio", "label"])

print("\nDataset splits:")
print({k: v.shape for k, v in dataset.items()})

print("\nDataset features:")
print(dataset["train"].features)


Dataset splits:
{'train': (352132, 2), 'val': (42198, 2), 'test': (44844, 2)}

Dataset features:
{'audio': List(List(List(Value('float64')))), 'label': ClassLabel(names=['other', 'drone'])}


In [5]:
train_test = dataset["train"].train_test_split(test_size=0.2, seed=SEED, stratify_by_column="label")
val_test = train_test["test"].train_test_split(test_size=0.5, seed=SEED, stratify_by_column="label")
dataset = DatasetDict({ "train": train_test["train"], "val": val_test["train"], "test": val_test["test"], })
print("\nDataset splits after train/val/test split:")
print({k: v.shape for k, v in dataset.items()})
# print(dataset["train"][1]["audio"].shape, dataset["train"][0]["label"])


Dataset splits after train/val/test split:
{'train': (281705, 2), 'val': (35213, 2), 'test': (35214, 2)}


In [6]:
def collate_fn(batch):
    """
    Prepares a batch for Conv2D model input.
    Returns tensors with shape (B, 1, 1, T).
    """
    xs = []
    ys = []

    for item in batch:
        audio_value = item["audio"]
        # Hugging Face Audio feature often gives {'array': np.ndarray, 'sampling_rate': ...}
        if isinstance(audio_value, dict) and "array" in audio_value:
            audio_value = audio_value["array"]
        # In some formats we may see a small wrapper object; try to fall back to '.array'
        elif hasattr(audio_value, "array"):
            audio_value = audio_value.array

        waveform = torch.as_tensor(audio_value, dtype=torch.float32)
        if waveform.ndim > 1:
            waveform = waveform.flatten()

        xs.append(waveform)
        ys.append(item["label"])

    xs = torch.nn.utils.rnn.pad_sequence(xs, batch_first=True)
    xs = xs.unsqueeze(1).unsqueeze(2)
    ys = torch.tensor(ys, dtype=torch.float32).unsqueeze(1)

    return xs, ys


def create_dataloaders(dataset_dict, batch_size, num_workers):
    train_loader = DataLoader(
        dataset_dict["train"],
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        collate_fn=collate_fn,
        pin_memory=True,
    )
    valid_loader = DataLoader(
        dataset_dict["val"],
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        collate_fn=collate_fn,
        pin_memory=True,
    )
    test_loader = DataLoader(
        dataset_dict["test"],
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        collate_fn=collate_fn,
        pin_memory=True,
    )
    return train_loader, valid_loader, test_loader


def waveform_to_model_tensor(waveform, sampling_rate=16000, target_sr=16000):
    """Convert waveform (np.ndarray or list) to model input shape (1, 1, 1, T)."""
    if sampling_rate != target_sr:
        waveform = librosa.resample(waveform, orig_sr=sampling_rate, target_sr=target_sr)
    x = torch.as_tensor(waveform, dtype=torch.float32)
    if x.ndim > 1:
        x = x.flatten()
    return x.unsqueeze(0).unsqueeze(0).unsqueeze(0)


train_loader, valid_loader, test_loader = create_dataloaders(
    dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS
)

print(f"\nCreated DataLoaders with Batch Size: {BATCH_SIZE}")

try:
    sample_x, sample_y = next(iter(train_loader))
    print(f"Sample batch shape - X: {sample_x.shape}, Y: {sample_y.shape}")
except Exception as e:
    print(f"Could not load a sample batch: {e}")


Created DataLoaders with Batch Size: 32
Sample batch shape - X: torch.Size([32, 1, 1, 4096]), Y: torch.Size([32, 1])


## Model Definition

In [7]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, kernel_size=3):
        super().__init__()
        padding = kernel_size // 2

        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.prelu = nn.PReLU()
        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=1,
            padding=padding,
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Identity()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        out = self.prelu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)
        out = self.prelu(out)
        return out


class AudioResNet(nn.Module):
    def __init__(
        self,
        channels=(32, 64, 128, 256),
        num_blocks=(1, 1, 1),
        dropout=0.5,
        fc_hidden=128,
        initial_kernel_size=5,
        block_kernel_size=3,
    ):
        super().__init__()

        if len(channels) != len(num_blocks) + 1:
            raise ValueError(
                "channels must have len(num_blocks) + 1 values: stem + one per stage."
            )

        stem_padding = initial_kernel_size // 2
        self.conv1 = nn.Conv2d(
            1,
            channels[0],
            kernel_size=initial_kernel_size,
            stride=(2, 1),
            padding=stem_padding,
        )
        self.bn1 = nn.BatchNorm2d(channels[0])
        self.prelu = nn.PReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        stages = []
        in_channels = channels[0]
        for stage_idx, stage_width in enumerate(channels[1:]):
            stage_blocks = []
            for block_idx in range(num_blocks[stage_idx]):
                stride = 2 if block_idx == 0 else 1
                stage_blocks.append(
                    ResidualBlock(
                        in_channels,
                        stage_width,
                        stride=stride,
                        kernel_size=block_kernel_size,
                    )
                )
                in_channels = stage_width
            stages.append(nn.Sequential(*stage_blocks))

        self.stages = nn.ModuleList(stages)
        self.pool2 = nn.AdaptiveAvgPool2d((1, 1))

        self.fc = nn.Sequential(
            nn.Linear(in_channels, fc_hidden),
            nn.PReLU(),
            nn.Dropout(dropout),
            nn.Linear(fc_hidden, 1),
        )

    def forward(self, x):
        x = self.pool1(self.prelu(self.bn1(self.conv1(x))))
        for stage in self.stages:
            x = stage(x)
        x = self.pool2(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


DEFAULT_MODEL_PARAMS = FALLBACK_MODEL_PARAMS.copy()

## Training Setup

In [8]:
@dataclass
class TrainResult:
    best_val_acc: float
    history: dict


def sample_model_params(trial):
    num_stages = trial.suggest_int("num_stages", 2, 4)
    stem_channels = trial.suggest_categorical("stem_channels", [16, 32, 48])

    stage_channels = []
    candidate_channels = [32, 64, 96, 128, 160, 192, 256]
    min_stage_width = stem_channels
    for stage_idx in range(num_stages):
        width = trial.suggest_categorical(
            f"stage_channels_{stage_idx}", candidate_channels
        )
        width = max(width, min_stage_width)
        stage_channels.append(width)
        min_stage_width = width

    channels = tuple([stem_channels] + stage_channels)

    blocks_per_stage = trial.suggest_int("num_blocks_per_stage", 1, 3)
    num_blocks = tuple([blocks_per_stage] * num_stages)

    model_params = {
        "channels": channels,
        "num_blocks": num_blocks,
        "dropout": trial.suggest_float("dropout", 0.1, 0.7),
        "fc_hidden": trial.suggest_categorical("fc_hidden", [64, 128, 256]),
        "initial_kernel_size": trial.suggest_categorical("initial_kernel_size", [3, 5, 7]),
        "block_kernel_size": trial.suggest_categorical("block_kernel_size", [3, 5, 7]),
    }
    return model_params


def train_and_validate(
    model,
    train_loader,
    valid_loader,
    epochs,
    learning_rate,
    weight_decay,
    trial=None,
    run_tag="train",
):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs))

    best_val_acc = 0.0
    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
    }

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", unit="batch")
        for x, y in train_pbar:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            out = model(x)
            loss = criterion(out, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            probs = torch.sigmoid(out)
            preds = (probs > 0.5).float()

            train_loss += loss.item() * x.size(0)
            train_correct += (preds == y).sum().item()
            train_total += y.numel()

            train_pbar.set_postfix(loss=loss.item())

        scheduler.step()

        epoch_train_loss = train_loss / len(train_loader.dataset)
        epoch_train_acc = train_correct / train_total

        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        valid_pbar = tqdm(valid_loader, desc=f"Epoch {epoch+1}/{epochs} [Valid]", unit="batch")
        with torch.no_grad():
            for x, y in valid_pbar:
                x = x.to(DEVICE, non_blocking=True)
                y = y.to(DEVICE, non_blocking=True)

                out = model(x)
                loss = criterion(out, y)

                probs = torch.sigmoid(out)
                preds = (probs > 0.5).float()

                val_loss += loss.item() * x.size(0)
                val_correct += (preds == y).sum().item()
                val_total += y.numel()

                valid_pbar.set_postfix(loss=loss.item())

        epoch_val_loss = val_loss / len(valid_loader.dataset)
        epoch_val_acc = val_correct / val_total

        history["train_loss"].append(epoch_train_loss)
        history["val_loss"].append(epoch_val_loss)
        history["train_acc"].append(epoch_train_acc)
        history["val_acc"].append(epoch_val_acc)

        LOGGER.info(
            "%s | epoch=%d/%d | train_loss=%.5f | val_loss=%.5f | train_acc=%.5f | val_acc=%.5f",
            run_tag,
            epoch + 1,
            epochs,
            epoch_train_loss,
            epoch_val_loss,
            epoch_train_acc,
            epoch_val_acc,
        )

        best_val_acc = max(best_val_acc, epoch_val_acc)

        if trial is not None:
            trial.report(epoch_val_acc, epoch)
            if trial.should_prune():
                LOGGER.info("%s | pruned at epoch=%d", run_tag, epoch + 1)
                raise optuna.exceptions.TrialPruned()

    return TrainResult(best_val_acc=best_val_acc, history=history)


def decode_audio_bytes(audio_raw):
    """Decode raw audio dict {'bytes': b'...', 'path': '...'} -> (float32 1-D array, sr).

    Works with Audio(decode=False) output, completely bypassing HF AudioDecoder.
    """
    audio_bytes = audio_raw.get("bytes")
    audio_path = audio_raw.get("path")

    if audio_bytes is not None:
        data, sr = sf.read(io.BytesIO(audio_bytes), dtype="float32")
    elif audio_path is not None:
        data, sr = sf.read(audio_path, dtype="float32")
    else:
        raise ValueError(f"Cannot decode audio: no bytes or path found. Keys: {list(audio_raw.keys())}")

    # stereo -> mono
    if data.ndim > 1:
        data = data.mean(axis=1)

    return data.astype(np.float32), sr


def audio_to_mel(data):
    """1-D float32 waveform -> normalised log-mel spectrogram (np.float32)."""
    if data.size > 0 and np.max(np.abs(data)) > 1.5:
        data = data / 32768.0
    mel = librosa.feature.melspectrogram(
        y=data, sr=SAMPLING_RATE, n_fft=1025, hop_length=256,
        n_mels=128, fmin=20, fmax=8000, power=2.0,
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)
    return mel_db.astype(np.float32)


class MelDataset(torch.utils.data.Dataset):
    """Simple in-memory dataset of (mel, label) pairs."""

    def __init__(self, mels, labels):
        self.mels = mels
        self.labels = labels

    def __len__(self):
        return len(self.mels)

    def __getitem__(self, idx):
        return self.mels[idx], self.labels[idx]


def collate_fn_all_tests(batch):
    """Collate for MelDataset: each item is (mel_tensor, label)."""
    xs = []
    ys = []
    for mel, label in batch:
        x = torch.as_tensor(mel, dtype=torch.float32)
        if x.ndim == 2:
            x = x.unsqueeze(0)  # (n_mels, time) -> (1, n_mels, time)
        xs.append(x.flatten())
        ys.append(label)

    xs = torch.nn.utils.rnn.pad_sequence(xs, batch_first=True)
    xs = xs.unsqueeze(1).unsqueeze(2)
    ys = torch.tensor(ys, dtype=torch.float32).unsqueeze(1)
    return xs, ys


def build_all_tests_dataloaders():
    LOGGER.info("Loading all-tests dataset: %s", ALL_TESTS_DATASET_NAME)
    raw_ds = load_dataset(ALL_TESTS_DATASET_NAME)

    # Disable HF AudioDecoder: get raw {'bytes': ..., 'path': ...} dicts
    for split in raw_ds:
        raw_ds[split] = raw_ds[split].cast_column(
            "audio", Audio(decode=False)
        )

    ds_all_tests = {}
    dataloaders = {}

    for split in raw_ds:
        LOGGER.info("Converting '%s' to mel spectrograms...", split)
        split_ds = raw_ds[split]
        mels = []
        labels = []

        for i in tqdm(range(len(split_ds)), desc=f"Mel {split}"):
            item = split_ds[i]
            data, _sr = decode_audio_bytes(item["audio"])  # soundfile: always float32
            mel = audio_to_mel(data)
            mels.append(mel)
            labels.append(int(item["label"]))

            if ALL_TESTS_MAX_SAMPLES_PER_SPLIT is not None and len(mels) >= ALL_TESTS_MAX_SAMPLES_PER_SPLIT:
                break

        mel_dataset = MelDataset(mels, labels)
        ds_all_tests[split] = mel_dataset
        dataloaders[split] = DataLoader(
            mel_dataset,
            batch_size=ALL_TESTS_BATCH_SIZE,
            shuffle=False,
            num_workers=0,  # no workers: data is already in memory
            collate_fn=collate_fn_all_tests,
        )
        LOGGER.info("all-tests split '%s' size=%d", split, len(mel_dataset))

    return ds_all_tests, dataloaders


def get_all_tests_dataloaders():
    """Always rebuild all_tests_ds_3 dataloaders (avoids stale caches)."""
    return build_all_tests_dataloaders()


def collect_probs_on_dataloaders(model, dataloaders):
    model.eval()
    results = {}

    with torch.no_grad():
        for split, loader in dataloaders.items():
            y_true, y_prob = [], []

            for x, y in loader:
                x = x.to(DEVICE)
                y = y.to(DEVICE)

                logits = model(x).squeeze(1)
                probs = torch.sigmoid(logits)

                y_prob.extend(probs.cpu().numpy().flatten())
                y_true.extend(y.cpu().numpy().flatten())

            results[split] = {"y_true": y_true, "y_prob": y_prob}

    return results


def score_from_probs(y_true, y_prob, threshold):
    y_true = np.array(y_true).astype(int)
    y_prob = np.array(y_prob).astype(float)
    y_pred = (y_prob > threshold).astype(int)
    return f1_score(y_true, y_pred, zero_division=0)


def find_best_threshold(results_payload, thresholds=None):
    if thresholds is None:
        thresholds = ALL_TESTS_THRESHOLD_GRID

    y_true_all = []
    y_prob_all = []

    for payload in results_payload.values():
        y_true_all.extend(payload["y_true"])
        y_prob_all.extend(payload["y_prob"])

    best_threshold = 0.5
    best_score = -1.0

    for threshold in thresholds:
        score = score_from_probs(y_true_all, y_prob_all, float(threshold))
        if score > best_score:
            best_score = score
            best_threshold = float(threshold)

    return best_threshold, best_score


def evaluate_all_tests_f1(model):
    _, dataloaders = get_all_tests_dataloaders()
    results_payload = collect_probs_on_dataloaders(model, dataloaders)
    best_threshold, best_f1 = find_best_threshold(results_payload)
    return best_f1, best_threshold, results_payload


def objective(trial):
    learning_rate = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-4, 1e-1, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    model_params = sample_model_params(trial)

    LOGGER.info(
        "trial=%d | start | lr=%.6g | wd=%.6g | batch_size=%d | model_params=%s",
        trial.number,
        learning_rate,
        weight_decay,
        batch_size,
        model_params,
    )

    try:
        train_loader_trial, valid_loader_trial, _ = create_dataloaders(
            dataset, batch_size=batch_size, num_workers=NUM_WORKERS
        )

        trial_model = AudioResNet(**model_params).to(DEVICE)
        train_result = train_and_validate(
            model=trial_model,
            train_loader=train_loader_trial,
            valid_loader=valid_loader_trial,
            epochs=OPTUNA_EPOCHS,
            learning_rate=learning_rate,
            weight_decay=weight_decay,
            trial=trial,
            run_tag=f"trial_{trial.number}",
        )

        all_tests_f1, all_tests_threshold, _ = evaluate_all_tests_f1(trial_model)
        trial.set_user_attr("all_tests_f1", float(all_tests_f1))
        trial.set_user_attr("all_tests_optimal_threshold", float(all_tests_threshold))
        trial.set_user_attr("val_acc_for_pruning", float(train_result.best_val_acc))

        LOGGER.info(
            "trial=%d | done | all_tests_f1=%.5f | threshold=%.2f | val_acc=%.5f",
            trial.number,
            all_tests_f1,
            all_tests_threshold,
            train_result.best_val_acc,
        )
        return float(all_tests_f1)
    except optuna.exceptions.TrialPruned:
        LOGGER.info("trial=%d | pruned", trial.number)
        raise
    except KeyboardInterrupt:
        LOGGER.warning("trial=%d | interrupted by user", trial.number)
        raise
    except Exception:
        LOGGER.exception("trial=%d | failed with exception", trial.number)
        raise

## Quick Sanity Check (small run)

Run this section first with a few samples and epochs to verify that training, evaluation on `all_tests_ds_3`, and logging all work before launching a long Optuna search.

In [9]:
# Small end-to-end check on a subset of the data.

QUICK_N_TRAIN = 512
QUICK_N_VAL = 128

small_dataset = DatasetDict(
    {
        "train": dataset["train"].select(
            range(min(QUICK_N_TRAIN, len(dataset["train"])))
        ),
        "val": dataset["val"].select(
            range(min(QUICK_N_VAL, len(dataset["val"])))
        ),
        "test": dataset["test"].select(
            range(min(QUICK_N_VAL, len(dataset["test"])))
        ),
    }
)

LOGGER.info(
    "quick_sanity | small_dataset sizes=%s",
    {k: len(v) for k, v in small_dataset.items()},
)

quick_train_loader, quick_val_loader, _ = create_dataloaders(
    small_dataset, batch_size=16, num_workers=NUM_WORKERS
)

quick_model = AudioResNet(**FALLBACK_MODEL_PARAMS).to(DEVICE)
quick_result = train_and_validate(
    model=quick_model,
    train_loader=quick_train_loader,
    valid_loader=quick_val_loader,
    epochs=1,
    learning_rate=FALLBACK_LR,
    weight_decay=FALLBACK_WEIGHT_DECAY,
    run_tag="quick_sanity",
)

quick_f1, quick_thr, _ = evaluate_all_tests_f1(quick_model)
print(
    f"Quick sanity | val_acc={quick_result.best_val_acc:.4f}, "
    f"all_tests_f1={quick_f1:.4f}, thr={quick_thr:.2f}"
)
LOGGER.info(
    "quick_sanity | done | val_acc=%.5f | all_tests_f1=%.5f | thr=%.2f",
    quick_result.best_val_acc,
    quick_f1,
    quick_thr,
)

2026-02-13 16:01:33 | INFO | quick_sanity | small_dataset sizes={'train': 512, 'val': 128, 'test': 128}
Epoch 1/1 [Valid]: 100%|██████████| 8/8 [00:00<00:00,  8.61batch/s, loss=1.34] 
2026-02-13 16:01:37 | INFO | quick_sanity | epoch=1/1 | train_loss=0.55028 | val_loss=0.99182 | train_acc=0.74414 | val_acc=0.50781
2026-02-13 16:01:37 | INFO | Loading all-tests dataset: Hibou-Foundation/all_tests_ds_3
2026-02-13 16:01:38 | INFO | Converting 'drone_test' to mel spectrograms...
Mel drone_test: 100%|██████████| 893/893 [00:02<00:00, 330.74it/s]
2026-02-13 16:01:41 | INFO | all-tests split 'drone_test' size=893
2026-02-13 16:01:41 | INFO | Converting 'drone_test_2' to mel spectrograms...
Mel drone_test_2: 100%|██████████| 2805/2805 [00:05<00:00, 482.04it/s]
2026-02-13 16:01:47 | INFO | all-tests split 'drone_test_2' size=2805
2026-02-13 16:01:47 | INFO | Converting 'parabole1' to mel spectrograms...
Mel parabole1: 100%|██████████| 123/123 [00:00<00:00, 453.57it/s]
2026-02-13 16:01:47 | INFO

Quick sanity | val_acc=0.5078, all_tests_f1=0.7730, thr=0.05


## Hyperparameter Search (Optuna, Optimizing F1 on all_tests_ds_3)

In [ ]:
# Run the "Training Setup" cell above first (defines objective).
def log_trial_callback(study, trial):
    value = trial.value if trial.value is not None else float("nan")
    LOGGER.info(
        "callback | trial=%d | state=%s | value=%.5f",
        trial.number,
        str(trial.state),
        value,
    )


pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1)
study = optuna.create_study(
    direction="maximize",
    pruner=pruner,
    study_name=OPTUNA_STUDY_NAME,
    storage=OPTUNA_STORAGE_URL,
    load_if_exists=True,
)

LOGGER.info(
    "Starting/Resuming Optuna study '%s' with n_trials=%d",
    OPTUNA_STUDY_NAME,
    N_TRIALS,
)

try:
    study.optimize(
        objective,
        n_trials=N_TRIALS,
        callbacks=[log_trial_callback],
        gc_after_trial=True,
    )
except KeyboardInterrupt:
    LOGGER.warning("Optimization interrupted. You can resume by rerunning this cell.")
    print("Optimization interrupted. Resume by rerunning this cell.")

completed_trials = [
    t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE
]

if not completed_trials:
    LOGGER.warning("No completed trials yet.")
    print("No completed trials yet.")
else:
    LOGGER.info(
        "Best trial so far | value=%.5f | params=%s",
        study.best_value,
        study.best_params,
    )
    print("Best F1 on all_tests_ds_3 from search:", study.best_value)
    print("Best params:", study.best_params)
    print("Best threshold:", study.best_trial.user_attrs.get("all_tests_optimal_threshold", 0.5))

print(f"Log file: {HPO_LOG_PATH}")
print(f"Optuna DB: {OPTUNA_STORAGE_PATH}")


[I 2026-02-13 16:04:37,172] Using an existing study with name 'resnet_custom_all_tests_f1' instead of creating a new one.
2026-02-13 16:04:37 | INFO | Starting/Resuming Optuna study 'resnet_custom_all_tests_f1' with n_trials=25
2026-02-13 16:04:37 | INFO | trial=1 | start | lr=0.000247201 | wd=0.0030297 | batch_size=64 | model_params={'channels': (16, 160, 160), 'num_blocks': (2, 2), 'dropout': 0.6915606354099125, 'fc_hidden': 256, 'initial_kernel_size': 5, 'block_kernel_size': 5}
Epoch 1/3 [Train]:   5%|▍         | 207/4402 [00:22<06:49, 10.25batch/s, loss=0.302]

In [ ]:
def best_params_to_model_params(best_params):
    num_stages = best_params["num_stages"]
    channels = [best_params["stem_channels"]]
    for stage_idx in range(num_stages):
        channels.append(best_params[f"stage_channels_{stage_idx}"])

    num_blocks = tuple([best_params["num_blocks_per_stage"]] * num_stages)

    return {
        "channels": tuple(channels),
        "num_blocks": num_blocks,
        "dropout": best_params["dropout"],
        "fc_hidden": best_params["fc_hidden"],
        "initial_kernel_size": best_params["initial_kernel_size"],
        "block_kernel_size": best_params["block_kernel_size"],
    }


completed_trials = []
if "study" in globals():
    completed_trials = [
        t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE
    ]

if completed_trials:
    best_params = study.best_params
    best_lr = best_params["lr"]
    best_weight_decay = best_params["weight_decay"]
    best_batch_size = best_params["batch_size"]
    best_model_params = best_params_to_model_params(best_params)
    best_optuna_all_tests_f1 = float(study.best_value)
    best_optuna_threshold = float(
        study.best_trial.user_attrs.get("all_tests_optimal_threshold", 0.5)
    )
else:
    LOGGER.warning("No completed Optuna trials found. Using fallback hyperparameters.")
    best_params = {}
    best_lr = FALLBACK_LR
    best_weight_decay = FALLBACK_WEIGHT_DECAY
    best_batch_size = BATCH_SIZE
    best_model_params = FALLBACK_MODEL_PARAMS
    best_optuna_all_tests_f1 = float("nan")
    best_optuna_threshold = 0.5

print("\nStarting final training...")
print("Model params:", best_model_params)
print(
    {
        "lr": best_lr,
        "weight_decay": best_weight_decay,
        "batch_size": best_batch_size,
        "epochs": EPOCHS,
    }
)
print(
    {
        "best_optuna_all_tests_f1": best_optuna_all_tests_f1,
        "best_optuna_threshold": best_optuna_threshold,
    }
)
LOGGER.info(
    "final_training | start | lr=%.6g | wd=%.6g | batch_size=%d | params=%s",
    best_lr,
    best_weight_decay,
    best_batch_size,
    best_model_params,
)

train_loader, valid_loader, test_loader = create_dataloaders(
    dataset, batch_size=best_batch_size, num_workers=NUM_WORKERS
)

model = AudioResNet(**best_model_params).to(DEVICE)
final_result = train_and_validate(
    model=model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    epochs=EPOCHS,
    learning_rate=best_lr,
    weight_decay=best_weight_decay,
    run_tag="final_training",
)

history = final_result.history
best_val_acc = final_result.best_val_acc

final_all_tests_f1, final_optimal_threshold, final_results_payload = evaluate_all_tests_f1(model)
OPTIMAL_THRESHOLD = final_optimal_threshold

os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "model_params": best_model_params,
        "best_params": best_params,
        "best_val_acc": best_val_acc,
        "best_optuna_all_tests_f1": best_optuna_all_tests_f1,
        "best_optuna_threshold": best_optuna_threshold,
        "final_all_tests_f1": float(final_all_tests_f1),
        "final_optimal_threshold": float(final_optimal_threshold),
    },
    MODEL_SAVE_PATH,
)

print(f"Saved model to {MODEL_SAVE_PATH}")
print(f"Best validation accuracy in final run: {best_val_acc:.4f}")
print(f"Final F1 on {ALL_TESTS_DATASET_NAME}: {final_all_tests_f1:.4f}")
print(f"Final optimal threshold on all_tests_ds_3: {final_optimal_threshold:.2f}")
LOGGER.info(
    "final_training | done | best_val_acc=%.5f | final_all_tests_f1=%.5f | threshold=%.2f | model_path=%s",
    best_val_acc,
    final_all_tests_f1,
    final_optimal_threshold,
    MODEL_SAVE_PATH,
)

## Training Visualization

In [ ]:
if history["train_loss"]:
    plt.style.use("ggplot")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

    ax1.plot(history["train_loss"], label="Train Loss")
    ax1.plot(history["val_loss"], label="Validation Loss")
    ax1.set_title("Loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()

    ax2.plot(history["train_acc"], label="Train Accuracy")
    ax2.plot(history["val_acc"], label="Validation Accuracy")
    ax2.set_title("Accuracy")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.legend()

    plt.tight_layout()
    plt.show()

## Model Loading and Test Evaluation

In [ ]:
LOAD_MODEL_PATH = MODEL_SAVE_PATH
print(f"Loading tuned model from {LOAD_MODEL_PATH}...")
checkpoint = torch.load(LOAD_MODEL_PATH, map_location=DEVICE)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    loaded_model_params = checkpoint.get("model_params", DEFAULT_MODEL_PARAMS)
    model = AudioResNet(**loaded_model_params).to(DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    OPTIMAL_THRESHOLD = float(
        checkpoint.get(
            "final_optimal_threshold",
            checkpoint.get("best_optuna_threshold", 0.5),
        )
    )
    print("Loaded model with stored architecture parameters.")
else:
    model = AudioResNet(**DEFAULT_MODEL_PARAMS).to(DEVICE)
    model.load_state_dict(checkpoint)
    OPTIMAL_THRESHOLD = 0.5
    print("Loaded legacy state_dict with default architecture.")

model.eval()
print(f"Model is ready for evaluation (threshold={OPTIMAL_THRESHOLD:.2f}).")

In [ ]:
all_preds = []
all_labels = []

threshold_for_eval = globals().get("OPTIMAL_THRESHOLD", 0.5)

with torch.no_grad():
    for x, y in tqdm(test_loader, desc="Testing"):
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        logits = model(x)
        probs = torch.sigmoid(logits)
        preds = (probs > threshold_for_eval).float()

        all_preds.extend(preds.cpu().numpy().flatten())
        all_labels.extend(y.cpu().numpy().flatten())

print(f"\nTest evaluation complete at threshold={threshold_for_eval:.2f}.")

In [ ]:
accuracy = accuracy_score(all_labels, all_preds)
print(f"\nTest Accuracy @thr={threshold_for_eval:.2f}: {accuracy*100:.2f}%")
target_names = dataset["train"].features["label"].names

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=target_names, digits=3))
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=[f"Predicted {target_names[0]}", f"Predicted {target_names[1]}"],
    yticklabels=[f"Actual {target_names[0]}", f"Actual {target_names[1]}"],
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

## Evaluation and Threshold Optimization on Custom Test Sets

In [ ]:
ds_all_tests, dataloaders = get_all_tests_dataloaders()
print({split: len(ds) for split, ds in ds_all_tests.items()})

In [ ]:
results = collect_probs_on_dataloaders(model, dataloaders)

In [ ]:
def compute_metrics(y_true, y_prob, threshold):
    y_true = np.array(y_true).astype(int)
    y_prob = np.array(y_prob).astype(float)
    y_pred = (y_prob > threshold).astype(int)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = float("nan")

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "auc": auc,
        "confusion_matrix": cm,
    }


if "OPTIMAL_THRESHOLD" in globals():
    threshold_for_metrics = OPTIMAL_THRESHOLD
else:
    threshold_for_metrics, _ = find_best_threshold(results)

print(
    "Best threshold on all_tests_ds_3 (f1): "
    f"{threshold_for_metrics:.2f}"
)

metrics = {}
for split, payload in results.items():
    metrics[split] = compute_metrics(
        payload["y_true"],
        payload["y_prob"],
        threshold=threshold_for_metrics,
    )

for split, metric in metrics.items():
    print(
        f"{split} @thr={threshold_for_metrics:.2f}: "
        f"acc={metric['accuracy']:.4f}, "
        f"prec={metric['precision']:.4f}, "
        f"rec={metric['recall']:.4f}, "
        f"f1={metric['f1']:.4f}, "
        f"auc={metric['auc']:.4f}"
    )

In [ ]:
CLASS_NAMES = ["other", "drone"]


def plot_confusion(cm, title):
    plt.figure(figsize=(4, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.tight_layout()
    plt.show()


for split, metric in metrics.items():
    plot_confusion(metric["confusion_matrix"], split)

## Custom Inference on WAV Files

In [ ]:
def infer_from_folder(folder_path: str, threshold=None):
    files = sorted([f for f in os.listdir(folder_path) if f.lower().endswith(".wav")])
    if not files:
        print("No .wav files found in folder.")
        return []

    if threshold is None:
        threshold = globals().get("OPTIMAL_THRESHOLD", 0.5)

    rows = []
    for file_name in files:
        file_path = os.path.join(folder_path, file_name)
        waveform, sr = librosa.load(file_path, sr=None, mono=True)

        x = waveform_to_model_tensor(waveform, sampling_rate=sr).to(DEVICE)

        with torch.no_grad():
            logits = model(x).squeeze(1)
            prob = torch.sigmoid(logits).cpu().item()
            pred = int(prob > threshold)

        rows.append(
            {
                "file": file_name,
                "predicted_label": pred,
                "confidence": prob,
            }
        )
        print(f"{file_name}: label={pred}, confidence={prob:.4f}")

    positive_ratio = sum(r["predicted_label"] for r in rows) / len(rows)
    print(f"Predicted drone ratio: {positive_ratio:.4f} (threshold={threshold:.2f})")
    return rows


print("Label mapping: 0=other, 1=drone")
print(f"Default inference threshold: {globals().get('OPTIMAL_THRESHOLD', 0.5):.2f}")
# Example:
# infer_from_folder("/absolute/path/to/wav_folder")